In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,SimpleRNN
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [43]:
with open("next_word_predictor.txt", encoding="utf-8") as f:
    text = f.read()

# Convert to lowercase
text = text.lower()
text=text.replace('\n\n','\n')
text=text.replace('\n\n\n','\n')

print("Total characters:", len(text))
print(text[:300])

Total characters: 165919
the sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. people were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. children were playing games, and laughter filled the air.
as t


In [44]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

Total words: 4994


In [45]:
input_sequences = []

for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    #print(tokenizer.texts_to_sequences([line])[0])
    #break
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])


In [46]:
max_len=max(len(seq) for seq in input_sequences)
print(max_len)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
print(input_sequences)

325
[[   0    0    0 ...    0    1  155]
 [   0    0    0 ...    1  155   21]
 [   0    0    0 ...  155   21 2368]
 ...
 [   0    0    0 ... 2331  290   19]
 [   0    0    0 ...  290   19   54]
 [   0    0    0 ...   19   54 1535]]


In [47]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
print(X)
print("\n\n",y)

[[   0    0    0 ...    0    0    1]
 [   0    0    0 ...    0    1  155]
 [   0    0    0 ...    1  155   21]
 ...
 [   0    0    0 ...   64 2331  290]
 [   0    0    0 ... 2331  290   19]
 [   0    0    0 ...  290   19   54]]


 [ 155   21 2368 ...   19   54 1535]


In [48]:
y=tf.keras.utils.to_categorical(y,num_classes=total_words)

In [49]:
model=Sequential()

In [50]:
model.add(Embedding(total_words,64,input_length=max_len-1))

In [51]:
model.add(SimpleRNN(128))

In [52]:
model.add(Dense(total_words,activation="softmax"))

In [53]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [54]:
model.fit(X,y,epochs=10,batch_size=64)

Epoch 1/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 64s 141ms/step - accuracy: 0.0452 - loss: 7.5361
Epoch 2/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 58s 140ms/step - accuracy: 0.0418 - loss: 7.4198
Epoch 3/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 57s 138ms/step - accuracy: 0.0539 - loss: 6.9623
Epoch 4/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 57s 139ms/step - accuracy: 0.0472 - loss: 7.3977
Epoch 5/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 57s 138ms/step - accuracy: 0.0613 - loss: 6.9724
Epoch 6/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 59s 142ms/step - accuracy: 0.0621 - loss: 6.9395
Epoch 7/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 58s 141ms/step - accuracy: 0.0754 - loss: 6.6960
Epoch 8/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 59s 143ms/step - accuracy: 0.0836 - loss: 6.5352
Epoch 9/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 59s 143ms/step - accuracy: 0.0971 - loss: 6.2344
Epoch 10/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 59s 142ms/step - accuracy: 0.1048 - loss: 6.0562


In [55]:
X.shape

(26383, 324)

In [56]:
y.shape

(26383, 4994)

In [57]:
def predict_next_word(model,tokenizer,text,max_len):
    token_list=tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list), axis=-1)
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

In [58]:
word=input("Enter the sentence: ")
print("Next word prediction: ",word,predict_next_word(model,tokenizer,word,max_len))

Enter the sentence:  the sun was


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 391ms/step
Next word prediction:  the sun was the


In [59]:
model.save("rnnmodel1.h5")